In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import requests
import pandas as pd

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from google.cloud import bigquery

# Configurações
PROJECT_ID = "macrodata-brasil"
TABELA_BRONZE = (
    f"{PROJECT_ID}.bronze.bcb_cambio_raw"
)

URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados"

DATA_INICIAL_HISTORICA = datetime(2024, 1, 1).date()
JANELA_REPROCESSAMENTO = 30

# Cliente BigQuery
client = bigquery.Client(
    project=PROJECT_ID,
    location="southamerica-east1"
)

# 1. Buscar última data carregada na Bronze
consulta = f"""
SELECT
    MAX(SAFE.PARSE_DATE('%d/%m/%Y', data)) AS ultima_data
FROM `{TABELA_BRONZE}`
"""

ultima_data = next(
    iter(client.query(consulta).result())
).ultima_data

# 2. Determinar janela incremental
hoje = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).date()

if ultima_data is None:
    data_inicial = DATA_INICIAL_HISTORICA
else:
    data_inicial = max(
        DATA_INICIAL_HISTORICA,
        ultima_data - timedelta(
            days=JANELA_REPROCESSAMENTO
        )
    )

print(f"Última data na Bronze: {ultima_data}")
print(f"Início da extração: {data_inicial}")
print(f"Fim da extração: {hoje}")

# 3. Buscar dados na API do Banco Central
parametros = {
    "formato": "json",
    "dataInicial": data_inicial.strftime("%d/%m/%Y"),
    "dataFinal": hoje.strftime("%d/%m/%Y")
}

resposta = requests.get(
    URL,
    params=parametros,
    timeout=30
)

resposta.raise_for_status()

# 4. Preparar DataFrame
df_cambio = pd.DataFrame(resposta.json())

df_cambio["data_extracao"] = datetime.now(timezone.utc)
df_cambio["fonte"] = "BCB_SGS_1"

print(f"Registros extraídos: {len(df_cambio)}")

display(df_cambio.tail())


Última data na Bronze: 2026-10-30
Início da extração: 2026-09-30
Fim da extração: 2026-11-17
Registros extraídos: 32


,data,valor,data_extracao,fonte
27,10/11/2026,5.5840,2026-11-17 12:33:58.089655+00:00,BCB_SGS_1
28,11/11/2026,5.5606,2026-11-17 12:33:58.089655+00:00,BCB_SGS_1
29,12/11/2026,5.6143,2026-11-17 12:33:58.089655+00:00,BCB_SGS_1
30,13/11/2026,5.6518,2026-11-17 12:33:58.089655+00:00,BCB_SGS_1
31,16/11/2026,5.7324,2026-11-17 12:33:58.089655+00:00,BCB_SGS_1


In [ ]:

import io
import json
import pandas as pd
from google.cloud import bigquery

TABELA_STAGING = (
    "macrodata-brasil.bronze.bcb_cambio_staging"
)

# 1. Validar a extração
if df_cambio.empty:
    raise ValueError(
        "Extração do câmbio vazia. Carga cancelada."
    )

registros = df_cambio[
    ["data", "valor", "data_extracao", "fonte"]
].copy()

# 2. Validar datas e valores
datas = pd.to_datetime(
    registros["data"],
    format="%d/%m/%Y",
    errors="coerce"
)

valores = pd.to_numeric(
    registros["valor"],
    errors="coerce"
)

if datas.isna().any() or valores.isna().any():
    raise ValueError(
        "Existem datas ou valores inválidos."
    )

if datas.duplicated().any():
    raise ValueError(
        "Existem datas duplicadas na extração."
    )

if (valores <= 0).any():
    raise ValueError(
        "Existem cotações inválidas (menores ou iguais a zero)."
    )

# 3. Padronizar tipos e timestamp
registros["data"] = registros["data"].astype(str)
registros["valor"] = registros["valor"].astype(str)

registros["data_extracao"] = (
    pd.to_datetime(
        registros["data_extracao"],
        utc=True
    ).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)

# 4. Converter para JSONL
conteudo_jsonl = "\n".join(
    json.dumps(registro, ensure_ascii=False)
    for registro in registros.to_dict(orient="records")
)

arquivo_memoria = io.BytesIO(
    conteudo_jsonl.encode("utf-8")
)

# 5. Configurar carregamento
configuracao = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("data", "STRING"),
        bigquery.SchemaField("valor", "STRING"),
        bigquery.SchemaField("data_extracao", "TIMESTAMP"),
        bigquery.SchemaField("fonte", "STRING"),
    ],
)

# 6. Carregar na staging
job = client.load_table_from_file(
    arquivo_memoria,
    TABELA_STAGING,
    job_config=configuracao,
    location="southamerica-east1",
)

job.result()

# 7. Validar o carregamento
tabela = client.get_table(TABELA_STAGING)

if tabela.num_rows != len(registros):
    raise ValueError(
        "Quantidade carregada diferente da extração."
    )

print("Staging do câmbio carregada com sucesso!")
print("Registros carregados:", tabela.num_rows)
print("Tabela:", TABELA_STAGING)
print("ID do job:", job.job_id)


Staging do câmbio carregada com sucesso!
Registros carregados: 32
Tabela: macrodata-brasil.bronze.bcb_cambio_staging
ID do job: 5362a70b-3131-44b7-9f1a-491c0712ac9b
